### Instruction

> 1. Rename assignment-01-XXX-YYY.ipynb where XXX is your student ID and YYY is your Chinese name.
> 2. The deadline of Assignment-01 is 23:59pm, 04-02-2026
> 3. In this assignment, you will
>    1) Explore Wikipedia text data
>    2) Build language models
>    3) Build NB and LR classifiers
>    4) Build embedding model for document classification
> 4. Submit your homework as html file just like our exercise did.
> Download the preprocessed data, enwiki-train.json and enwiki-test.json from the Assignment-01 folder. In the data file, each line contains a Wikipedia page with attributes, title, label, and text. There are 9000 records in the train file and 1000 records in test file with ten categories.

- Please print out all your outputs in the notebook and save it.

### Task1 - Data exploring

> 1) Print out how many documents are in each class  (for both train and test dataset)

In [1]:
# Your code goes to here

import json
from collections import Counter

# Function to load json lines data
def load_data(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data     # a list of dicts, each dict with keys: "title", "label", and "text"

# Load datasets
train_data = load_data('./.datasets/enwiki-train.json')
test_data = load_data('./.datasets/enwiki-test.json')

# Count labels in train set
train_labels = [item['label'] for item in train_data]
train_counter = Counter(train_labels)

# Count labels in test set
test_labels = [item['label'] for item in test_data]
test_counter = Counter(test_labels)

# Print results
print("Train dataset class distribution:")
for label, count in train_counter.items():
    print(f"{label}: {count}")

print("\nTest dataset class distribution:")
for label, count in test_counter.items():
    print(f"{label}: {count}")

Train dataset class distribution:
Book: 858
Food: 121
Film: 2752
Politician: 3441
Animal: 82
Writer: 769
Artist: 457
Disease: 202
Actor: 79
Software: 239

Test dataset class distribution:
Politician: 383
Writer: 68
Book: 117
Film: 296
Artist: 63
Actor: 1
Food: 16
Disease: 18
Software: 27
Animal: 11


> 2) Print out the average number of sentences in each class.
>    You may need to use sentence tokenization tools from spacy for both train and test dataset.



In [2]:
# Your code goes to here

import spacy
from tqdm import tqdm
from collections import defaultdict
import re

# Load English tokenizer from spaCy
# (Note: run "python -m spacy download en_core_web_sm" first)
nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "ner", "lemmatizer"])
nlp.add_pipe("sentencizer")


def process_dataset(data, desc="Processing"):
    '''
    compute average number of sentences AND tokens per class
    remove punctuations and other special characters, and make all words as lower cases for each sentence
    '''
    # stats
    sentence_counts = defaultdict(list)
    token_counts = defaultdict(list)

    # processed data
    sentences_by_class = defaultdict(list)
    all_sentences = []

    # store first article per class
    first_article_output = {}

    for item in tqdm(data, desc=desc):
        label = item['label']
        text = item['text']
        title = item['title']

        # Use spaCy to process text and split sentences
        doc = nlp(text)

        # Sentence count
        num_sentences = len(list(doc.sents))
        sentence_counts[label].append(num_sentences)

        # Token count
        num_tokens = len(doc)
        token_counts[label].append(num_tokens)

        # Process sentences
        cleaned_words_for_first = []

        for sent in doc.sents:
            cleaned_sentence = []

            for token in sent:
                tok = token.text.lower()

                # keep only English words and numbers
                if re.fullmatch(r"[a-z0-9]+", tok):
                    cleaned_sentence.append(tok)

                    # collect for "first 40 words"
                    if label not in first_article_output.keys():
                        if len(cleaned_words_for_first) < 40:
                            cleaned_words_for_first.append(tok)

            # skip empty sentences
            if len(cleaned_sentence) > 0:
                sentences_by_class[label].append(cleaned_sentence)
                all_sentences.append(cleaned_sentence)

        # Store first article output
        if label not in first_article_output.keys():
            first_article_output[label] = {
                "title": title,
                "words": cleaned_words_for_first
            }

    # Compute averages
    avg_sentences = {}
    avg_tokens = {}
    
    for label in sentence_counts.keys():
        avg_sentences[label] = sum(sentence_counts[label]) / len(sentence_counts[label])
        avg_tokens[label] = sum(token_counts[label]) / len(token_counts[label])

    return avg_sentences, avg_tokens, sentences_by_class, all_sentences, first_article_output


# Process both train and test datasets
train_avg_sent, train_avg_tokens, train_sent_by_class, train_all_sents, train_first = process_dataset(
    train_data, desc="Processing train dataset"
)

test_avg_sent, test_avg_tokens, test_sent_by_class, test_all_sents, test_first = process_dataset(
    test_data, desc="Processing test dataset"
)

# Print results
print("Average number of sentences per class (Train):")
for label, avg in sorted(train_avg_sent.items()):
    print(f"{label}: {avg:.2f}")

print("\nAverage number of sentences per class (Test):")
for label, avg in sorted(test_avg_sent.items()):
    print(f"{label}: {avg:.2f}")

Processing test dataset: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [04:44<00:00,  3.51it/s]

Average number of sentences per class (Train):
Actor: 69.33
Animal: 67.06
Artist: 185.94
Book: 205.09
Disease: 347.86
Film: 177.88
Food: 153.44
Politician: 222.89
Software: 201.13
Writer: 216.13

Average number of sentences per class (Test):
Actor: 177.00
Animal: 62.82
Artist: 174.97
Book: 197.96
Disease: 325.89
Film: 173.69
Food: 165.12
Politician: 233.05
Software: 204.00
Writer: 220.62


> 3) Print out the average number of tokens in each class for both train and test dataset.

In [3]:
# Your code goes to here

print("Average number of tokens per class (Train):")
for label, avg in sorted(train_avg_tokens.items()):
    print(f"{label}: {avg:.2f}")

print("\nAverage number of tokens per class (Test):")
for label, avg in sorted(test_avg_tokens.items()):
    print(f"{label}: {avg:.2f}")

Average number of tokens per class (Train):
Actor: 1737.71
Animal: 1490.15
Artist: 4970.04
Book: 5445.62
Disease: 8328.23
Film: 4577.45
Food: 3594.41
Politician: 5844.88
Software: 5017.71
Writer: 5944.18

Average number of tokens per class (Test):
Actor: 4468.00
Animal: 1366.91
Artist: 4656.83
Book: 5251.16
Disease: 8142.22
Film: 4476.76
Food: 3661.75
Politician: 6115.95
Software: 4972.74
Writer: 6103.44


> 4) For each sentence in the document, remove punctuations and other special characters so that each sentence only contains English words and numbers. To make your life easier, you can make all words as lower cases. For each class, print out the first article's name and the processed first 40 words. (for both train and test dataset)

In [4]:
# Your code goes to here

import os

# Create directory if not exists
os.makedirs(".datasets/processed", exist_ok=True)

# Save functions
def save_all_sentences(sentences, path):
    with open(path, "w", encoding="utf-8") as f:
        for sent in sentences:
            f.write(json.dumps(sent) + "\n")

def save_sentences_by_class(data, path):
    # turn defaultdict to dict
    data = {label: sents for label, sents in data.items()}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f)

# Save train and test processed sentences
save_all_sentences(train_all_sents, "./.datasets/processed/train_all_sentences.json")
save_all_sentences(test_all_sents, "./.datasets/processed/test_all_sentences.json")
save_sentences_by_class(train_sent_by_class, "./.datasets/processed/train_by_class.json")
save_sentences_by_class(test_sent_by_class, "./.datasets/processed/test_by_class.json")

# Print function
def print_first_articles(first_articles, name="Dataset"):
    print(f"\n{name}:\n")
    for label, info in first_articles.items():
        print(f"Class: {label}")
        print(f"Title: {info['title']}")
        print("First 40 processed words:")
        print(" ".join(info['words']))
        print("-" * 50)

# For each class, print out the first article's name and the processed first 40 words
print_first_articles(train_first, name="Train dataset")
print_first_articles(test_first, name="Test dataset")


Train dataset:

Class: Book
Title: Middlesex_(novel)
First 40 processed words:
middlesex is a pulitzer prize winning novel by jeffrey eugenides published in 2002 the book is a bestseller with more than four million copies sold since its publication its characters and events are loosely based on aspects of eugenides life
--------------------------------------------------
Class: Food
Title: Chowder
First 40 processed words:
chowder is a type of soup or stew often prepared with milk or cream and thickened with broken crackers crushed ship biscuit or a roux variations of chowder can be seafood or vegetable crackers such as oyster crackers or saltines
--------------------------------------------------
Class: Film
Title: Young_People_Fucking
First 40 processed words:
young people fucking distributed as ypf in us and uk markets is a 2008 canadian sex comedy film directed by martin gero who co wrote it with aaron abrams the film story is told in a linear fashion alternating through
----------

### Task2 - Build language models

Before you go, you should do necessary preprocessing for training and testing text. For example, you should do sentence tokenization, removing special characters, replacing less frequency words as UNK (for example, you can try to use a cutoff of 10), making all words as lower characters. Fix your vocabulary size so that is not too large.

> 1) Based on the training dataset (collect all sentences in training dataset), build unigram, bigram, and trigram language models using Additive smoothing technique. It is encouraged to implement models by yourself.


In [1]:
# Your code goes to here

'''
Before we go, we first do necessary preprocessing for training and testing text.
We reuse the cleaned sentences from Task 1 and perform additional preprocessing for language modeling:

1. Build vocabulary from training data only.
2. Replace low-frequency words (frequency < 10) with <UNK>.
3. Add sentence boundary tokens <s> and </s> to each sentence.
'''

import json
from collections import Counter
from tqdm import tqdm
import os

# Config
UNK_TOKEN = "<UNK>"
START_TOKEN = "<s>"
END_TOKEN = "</s>"
MIN_FREQ = 10         # cutoff threshold

# Load processed data
def load_all_sentences(path):
    sentences = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            sentences.append(json.loads(line))
    return sentences       # a list of lists of tokens

train_sentences = load_all_sentences("./.datasets/processed/train_all_sentences.json")
test_sentences = load_all_sentences("./.datasets/processed/test_all_sentences.json")

# Build vocabulary from train dataset
word_counter = Counter()

for sent in train_sentences:
    word_counter.update(sent)

# Keep words with freq >= MIN_FREQ
vocab = {word for word, freq in word_counter.items() if freq >= MIN_FREQ}

# Add special tokens
vocab.update([UNK_TOKEN, START_TOKEN, END_TOKEN])

print(f"Vocabulary size (after cutoff={MIN_FREQ}): {len(vocab)}")


# Replace UNK AND add sentence boundaries
def process_sentences(sentences, vocab):
    processed = []

    for sent in tqdm(sentences, desc="Processing sentences"):
        new_sent = []

        # Replace low-frequency words
        for word in sent:
            if word in vocab:
                new_sent.append(word)
            else:
                new_sent.append(UNK_TOKEN)

        # Add sentence boundary tokens
        new_sent = [START_TOKEN] + new_sent + [END_TOKEN]

        processed.append(new_sent)

    return processed


train_processed = process_sentences(train_sentences, vocab)
test_processed = process_sentences(test_sentences, vocab)


# Save processed data
os.makedirs("./.datasets/lm_preprocessed", exist_ok=True)

# Save the vocabulary
def save_vocab(vocab, path):
    with open(path, "w", encoding="utf-8") as f:
        for word in vocab:
            f.write(word + "\n")

save_vocab(vocab, "./.datasets/lm_preprocessed/vocab.txt")

# Save word frequencies
def save_word_freq(word_counter, path):
    with open(path, "w", encoding="utf-8") as f:
        for word, counter in word_counter.items():
            f.write(f"{word}\t{counter}\n")

save_word_freq(word_counter, "./.datasets/lm_preprocessed/word_freq.txt")

# Save processed sentences
def save_sentences(sentences, path):
    with open(path, "w", encoding="utf-8") as f:
        for sent in sentences:
            f.write(json.dumps(sent) + "\n")

save_sentences(train_processed, "./.datasets/lm_preprocessed/train_lm.json")
save_sentences(test_processed, "./.datasets/lm_preprocessed/test_lm.json")

print("Preprocessed data saved.")

Vocabulary size (after cutoff=10): 73376


Processing sentences: 100%|█████████████████████████████████████████████████████████████████████████████████████| 204552/204552 [00:02<00:00, 74101.87it/s]


Preprocessed data saved.


In [ ]:
# Your code goes to here



> 2) Report the perplexity of these 3 trained models on the testing dataset (again collect all sentences in training dataset) and explain your findings. 

In [ ]:
# Your code goes to here




> 3) Use each built model to generate five sentences and explain these generated patterns.


In [ ]:
# Your code goes to here




> 4) Train bigram and trigram model using kenlm and report the perplexities of these two. Compare results of your model and results from kenlm

In [ ]:
# Your code goes to here




## Task3 - Build NB/LR classifiers

> 1) Build a Naive Bayes classifier (with Laplace smoothing) and test your model on test dataset

In [ ]:
# Your code goes to here




> 2) Build a LR classifier. This question seems to be challenging. We did not directly provide features for samples. But just use your own method to build useful features. You may need to split the training dataset into train and validation so that some involved parameters can be tuned. 

In [ ]:
# Your code goes to here




> 3) Report Micro-F1 score and Macro-F1 score for these classifiers on testing dataset explain your results.

In [ ]:
# Your code goes to here




### Task 4 - Classify documents using embeddings

> For each document (both training and testing documents), you have several choices to generate a document embedding from the embedding we trained in Task 1 (**Just choose one of them**):

> - Use the average of known static embeddings of all words in each document. Or use the first paragraph’s words and take an average on these embeddings.
> - Use the doc2vec algorithm to present each document.
> - Use modern embedding like Qwen-embedding 0.6b ...

> Build a classifer on training documents and testing this classifer on testing documents. Since you have the ground truth, you can use the Micro/Macro F1-score to measure the performance of these choices on testing documents.

In [ ]:
# Your code goes to here




#### upload your homework: use html file format
!jupyter nbconvert assignment-01-XXX-YYY.ipynb --to html --template classic --embed-images